In [ ]:
from pathlib import Path
import io, sys, subprocess, tempfile, zipfile
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one Lab2_colab.zip")
root = Path(tempfile.mkdtemp(prefix="lab2_", dir="/content"))
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for name in archive.namelist():
        if not (root / name).resolve().is_relative_to(root.resolve()):
            raise ValueError("Unsafe archive entry")
    archive.extractall(root)
PROJECT_ROOT = root
if not (PROJECT_ROOT / "requirements.txt").exists():
    raise ValueError("Use the ZIP produced by scripts/package_colab.py")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)


In [ ]:
import torch
from google.colab import drive
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime")
print("GPU:", torch.cuda.get_device_name(0))
drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/Lab2/outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
GPU_SMOKE_VERIFIED = False
subprocess.run([
    sys.executable, str(PROJECT_ROOT / "scripts/run_experiments.py"),
    "--configs", str(PROJECT_ROOT / "configs/selected.json"),
    "--epochs", "2", "--device", "cuda", "--output-dir", str(OUTPUT_ROOT / "smoke"),
], cwd=PROJECT_ROOT, check=True)
GPU_SMOKE_VERIFIED = True


In [ ]:
RUN_FULL_EXPERIMENTS = False
CONFIGS = [PROJECT_ROOT / "configs/selected.json"]
MODEL_SEEDS = [42, 43, 44]
if RUN_FULL_EXPERIMENTS:
    if not GPU_SMOKE_VERIFIED:
        raise RuntimeError("Complete GPU verification first")
    subprocess.run([
        sys.executable, str(PROJECT_ROOT / "scripts/run_experiments.py"),
        "--configs", *map(str, CONFIGS), "--seeds", *map(str, MODEL_SEEDS),
        "--device", "cuda", "--output-dir", str(OUTPUT_ROOT / "experiments"),
    ], cwd=PROJECT_ROOT, check=True)
else:
    print("Full training disabled. Reviewed results are already saved in the repository.")


In [ ]:
import shutil
if RUN_FULL_EXPERIMENTS:
    archive = shutil.make_archive("/content/Lab2_results", "zip", root_dir=OUTPUT_ROOT / "experiments")
    files.download(archive)


In [ ]:
from pathlib import Path
import subprocess
import sys
import shutil
import torch
from google.colab import files
from IPython.display import display, Image

if not all(name in globals() for name in ("PROJECT_ROOT", "OUTPUT_ROOT")):
    raise RuntimeError("Run setup sections 1 and 2 first")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime")
if not str(Path(OUTPUT_ROOT).resolve()).startswith("/content/drive/"):
    raise RuntimeError("Use Google Drive storage for full experiments")
BONUS_ROOT = Path(OUTPUT_ROOT) / "bonus_deepconvnet"
subprocess.run([
    sys.executable, str(PROJECT_ROOT / "scripts/run_bonus.py"),
    "--smoke", "--device", "cuda", "--output-dir", str(OUTPUT_ROOT / "bonus_smoke"),
], cwd=PROJECT_ROOT, check=True)
subprocess.run([
    sys.executable, str(PROJECT_ROOT / "scripts/run_bonus.py"),
    "--device", "cuda", "--output-dir", str(BONUS_ROOT),
], cwd=PROJECT_ROOT, check=True)
print((BONUS_ROOT / "seed_summary.csv").read_text())
print((BONUS_ROOT / "comparison.csv").read_text())
print("Selected-checkpoint test scores and monitored test maxima are separate columns.")
print("Take a screenshot of the summary if needed. Return the ZIP for independent review.")
display(Image(filename=str(BONUS_ROOT / "figures/eegnet_deepconvnet_comparison.png")))
archive = shutil.make_archive("/content/Lab2_bonus_results", "zip", root_dir=BONUS_ROOT)
files.download(archive)
